In [ ]:
# ── CELLULE 1 : Chargement des données ──
import pandas as pd
import os, csv
from sklearn.preprocessing import LabelEncoder

Proj = "PRJNA755688-stage12"
path = "/mnt/g/ngs/data/" + Proj
path1 = path + "/"
path2 = path + "/rna/"
fichiers_non_trouve = []

f = path1 + "liste-files.csv"
df_tot = pd.DataFrame()

with open(f, 'r') as csvfile:
    filereader = csv.reader(csvfile, delimiter=',')
    for row in filereader:
        sample = row[0]
        label = row[1]

        f1 = path2 + "results-" + sample + ".txt"
        if os.path.isfile(f1):
            df = pd.read_csv(f1, sep="\t")
            df["sample"] = sample
            df_pivot = df.pivot_table(values=['count'], columns='RNA', index=['sample'])

            if "I" in label:
                label = "II"
            if "healthy" in label:
                label = "control"
            if "Normal" in label:
                label = "control"

            df_pivot["label"] = label
            df_tot = pd.concat([df_tot, df_pivot])
        else:
            fichiers_non_trouve.append(f1)

print("Fichiers non trouvés :", len(fichiers_non_trouve))

encoder = LabelEncoder()
print(df_tot.label.unique())
print(df_tot.label.value_counts())

target = encoder.fit_transform(df_tot.label)
df_tot = df_tot.fillna(0)

Fichiers non trouvés : 0
['II' 'control']
label
control    561
II         399
Name: count, dtype: int64


In [3]:
# ── CELLULE 2 : Vérification labels ──
print(df_tot["label"])

sample
SRR17005923         II
SRR17005924         II
SRR17005930         II
SRR17005931         II
SRR17005945         II
                ...   
SRR16919803    control
SRR16919804    control
SRR16919805    control
SRR16919806    control
SRR16919807    control
Name: label, Length: 960, dtype: object


In [4]:
# ── CELLULE 3 : Vérification target ──
print(target)
print(target.shape)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 

In [5]:
# ── CELLULE 4 : Équilibrage des classes ──
from random import sample
def weights_balanced(col):
    min = col.value_counts().min()
    weights = col.value_counts()
    weights_balanced = {}
    for i, val in enumerate(weights):
        key = weights.index[i]
        weights_balanced[key] = str(min / val)
    return weights_balanced

my_weights_balanced = weights_balanced(df_tot.label)
df_balanced = pd.DataFrame()
for var in df_tot["label"].unique():
    df_balanced = pd.concat([df_balanced,
        df_tot.loc[df_tot['label'] == var].sample(frac=float(my_weights_balanced[var]))], axis=0)

df_tot = df_balanced
target = encoder.fit_transform(df_tot.label)
df_tot = df_tot.fillna(0)
print(df_tot.label.value_counts())
print(target)

label
II         399
control    399
Name: count, dtype: int64
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [6]:
# ── CELLULE 5 : NCA + PCA (facultatif) ──
from sklearn.neighbors import NeighborhoodComponentsAnalysis
import numpy as np
from sklearn.decomposition import PCA

try:
    df_tot["label"]
except KeyError:
    print("Colum label not defined")
else:
    df_tot = df_tot.drop(columns=["label"])

nca = NeighborhoodComponentsAnalysis(random_state=42)
nca.fit(df_tot, target)
pca = PCA(n_components=None, random_state=42).fit(nca.transform(df_tot))
eigenvalues = pca.explained_variance_ratio_
cumulative_sum = np.cumsum(eigenvalues)
variance_explained = cumulative_sum / np.sum(eigenvalues)
threshold = 1
n_components = np.argmax(variance_explained >= threshold) + 1
selected_vars = nca.transform(df_tot)[:, :n_components]
print(selected_vars.shape)

(798, 1)


In [7]:
# ── CELLULE 6 : Dimensions PCA ──
print(selected_vars.shape)
print(target.shape)

(798, 1)
(798,)


In [9]:
# ── CELLULE 7 : ML sklearn (main) ──
import numpy as np
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from imblearn.metrics import specificity_score
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    df_tot["label"]
except KeyError:
    print("Colum label not defined")
else:
    df_tot = df_tot.drop(columns=["label"])

X_train, X_test, y_train, y_test = train_test_split(df_tot, target, test_size=0.20, random_state=222)

print(y_test)
num_train_classes = np.unique(y_train).size
if num_train_classes < 2:
    print(f"Cannot train classifiers because training data contains only one class: {np.unique(y_train)}")
else:
    logreg = LogisticRegression(max_iter=10000)
    logreg.fit(X_train, y_train)
    y_pred_test_logreg = logreg.predict(X_test)
    cm = confusion_matrix(y_test, y_pred_test_logreg)
    print(cm)
    print(accuracy_score(y_test, y_pred_test_logreg))

    clf = SVC(gamma='scale', class_weight='balanced')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(accuracy_score(y_test, y_pred_test_logreg))

    from sklearn.naive_bayes import GaussianNB
    gnb = GaussianNB()
    gnb.fit(X_train, y_train)
    y_pred = gnb.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(accuracy_score(y_test, y_pred))

    rf_model = RandomForestClassifier(n_estimators=50, max_features="sqrt", random_state=44)
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print("Forets aleatoires:")
    print(cm)
    print(classification_report(y_test, y_pred))
    print("Specificite:")
    print(specificity_score(y_test, y_pred, average='weighted'))

    model = KNeighborsClassifier(n_neighbors=3)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(accuracy_score(y_test, y_pred))

    clfb = BalancedRandomForestClassifier()
    clfb.fit(X_train, y_train)
    y_pred = clfb.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print("Balanced Forets aleatoires:")
    print(cm)
    print(accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print("Specificite:")
    print(specificity_score(y_test, y_pred, average='weighted'))

    clf = SVC(C=0.1, gamma=1, kernel='poly')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print("SVC opti:")
    print(cm)
    print(classification_report(y_test, y_pred))

    print("Foret aleatoire optimis:")
    rf_model = RandomForestClassifier(n_estimators=200, max_depth=2, criterion='entropy')
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    print("Specificite:")
    print(specificity_score(y_test, y_pred, average='weighted'))
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(classification_report(y_test, y_pred))

Colum label not defined
[1 1 1 1 0 0 0 1 0 1 0 0 1 0 1 0 1 0 0 1 0 1 1 1 1 0 1 1 0 0 1 0 0 0 0 0 0
 1 0 1 1 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0 0 0 0
 0 1 1 1 1 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 1 1 1 0 0 0 0 1 0 0 0 1
 1 1 0 0 1 1 1 0 1 1 1 1 0 1 0 1 1 0 1 0 0 1 0 0 0 1 0 0 0 1 0 1 1 0 0 0 1
 0 1 1 0 1 1 1 1 1 1 1 1]
[[82  0]
 [ 1 77]]
0.99375
[[82  0]
 [ 0 78]]
0.99375
[[82  0]
 [ 0 78]]
1.0
Forets aleatoires:
[[82  0]
 [ 0 78]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.00      1.00        78

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160

Specificite:
1.0
[[82  0]
 [ 0 78]]
1.0
Balanced Forets aleatoires:
[[82  0]
 [ 0 78]]
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
        

In [10]:
# ── CELLULE 8 : ML sklearn (sans PCA, b) ──
import numpy as np
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from imblearn.metrics import specificity_score
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

X_train, X_test, y_train, y_test = train_test_split(df_tot, target, test_size=0.20, random_state=222)

print(y_test)
logreg = LogisticRegression(max_iter=10000)
logreg.fit(X_train, y_train)
y_pred_test_logreg = logreg.predict(X_test)
cm = confusion_matrix(y_test, y_pred_test_logreg)
print(cm)
print(accuracy_score(y_test, y_pred_test_logreg))

clf = SVC(gamma='scale', class_weight='balanced')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(accuracy_score(y_test, y_pred_test_logreg))

from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(accuracy_score(y_test, y_pred))

rf_model = RandomForestClassifier(n_estimators=50, max_features="sqrt", random_state=44)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("Forets aleatoires:")
print(cm)
print(classification_report(y_test, y_pred))
print("Specificite:")
print(specificity_score(y_test, y_pred, average='weighted'))

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(accuracy_score(y_test, y_pred))

clfb = BalancedRandomForestClassifier()
clfb.fit(X_train, y_train)
y_pred = clfb.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("Balanced Forets aleatoires:")
print(cm)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Specificite:")
print(specificity_score(y_test, y_pred, average='weighted'))

clf = SVC(C=0.1, gamma=1, kernel='poly')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("SVC opti:")
print(cm)
print(classification_report(y_test, y_pred))

print("Foret aleatoire optimis:")
rf_model = RandomForestClassifier(n_estimators=200, max_depth=2, criterion='entropy')
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
print("Specificite:")
print(specificity_score(y_test, y_pred, average='weighted'))
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred))

[1 1 1 1 0 0 0 1 0 1 0 0 1 0 1 0 1 0 0 1 0 1 1 1 1 0 1 1 0 0 1 0 0 0 0 0 0
 1 0 1 1 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0 0 0 0
 0 1 1 1 1 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 1 1 1 0 0 0 0 1 0 0 0 1
 1 1 0 0 1 1 1 0 1 1 1 1 0 1 0 1 1 0 1 0 0 1 0 0 0 1 0 0 0 1 0 1 1 0 0 0 1
 0 1 1 0 1 1 1 1 1 1 1 1]
[[82  0]
 [ 1 77]]
0.99375
[[82  0]
 [ 0 78]]
0.99375
[[82  0]
 [ 0 78]]
1.0
Forets aleatoires:
[[82  0]
 [ 0 78]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.00      1.00        78

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160

Specificite:
1.0
[[82  0]
 [ 0 78]]
1.0
Balanced Forets aleatoires:
[[82  0]
 [ 0 78]]
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.0

In [11]:
# ── CELLULE 9 : ML sklearn (sans PCA, c) ──
import numpy as np
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from imblearn.metrics import specificity_score
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

X_train, X_test, y_train, y_test = train_test_split(df_tot, target, test_size=0.20, random_state=222)

print(y_test)
logreg = LogisticRegression(max_iter=10000)
logreg.fit(X_train, y_train)
y_pred_test_logreg = logreg.predict(X_test)
cm = confusion_matrix(y_test, y_pred_test_logreg)
print(cm)
print(accuracy_score(y_test, y_pred_test_logreg))

clf = SVC(gamma='scale', class_weight='balanced')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(accuracy_score(y_test, y_pred_test_logreg))

from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(accuracy_score(y_test, y_pred))

rf_model = RandomForestClassifier(n_estimators=50, max_features="sqrt", random_state=44)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("Forets aleatoires:")
print(cm)
print(classification_report(y_test, y_pred))
print("Specificite:")
print(specificity_score(y_test, y_pred, average='weighted'))

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(accuracy_score(y_test, y_pred))

clfb = BalancedRandomForestClassifier()
clfb.fit(X_train, y_train)
y_pred = clfb.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("Balanced Forets aleatoires:")
print(cm)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Specificite:")
print(specificity_score(y_test, y_pred, average='weighted'))

clf = SVC(C=0.1, gamma=1, kernel='poly')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("SVC opti:")
print(cm)
print(classification_report(y_test, y_pred))

print("Foret aleatoire optimis:")
rf_model = RandomForestClassifier(n_estimators=200, max_depth=2, criterion='entropy')
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
print("Specificite:")
print(specificity_score(y_test, y_pred, average='weighted'))
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred))

[1 1 1 1 0 0 0 1 0 1 0 0 1 0 1 0 1 0 0 1 0 1 1 1 1 0 1 1 0 0 1 0 0 0 0 0 0
 1 0 1 1 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0 0 0 0
 0 1 1 1 1 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 1 1 1 0 0 0 0 1 0 0 0 1
 1 1 0 0 1 1 1 0 1 1 1 1 0 1 0 1 1 0 1 0 0 1 0 0 0 1 0 0 0 1 0 1 1 0 0 0 1
 0 1 1 0 1 1 1 1 1 1 1 1]
[[82  0]
 [ 1 77]]
0.99375
[[82  0]
 [ 0 78]]
0.99375
[[82  0]
 [ 0 78]]
1.0
Forets aleatoires:
[[82  0]
 [ 0 78]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.00      1.00        78

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160

Specificite:
1.0
[[82  0]
 [ 0 78]]
1.0
Balanced Forets aleatoires:
[[82  0]
 [ 0 78]]
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        82
           1       1.00      1.0